In [1]:
# Установка зависимостей (запустить один раз)
!pip install faiss-cpu sentence-transformers --quiet


# ====================== HW14 – Эмбеддинги, FAISS, оценка retrieval и mini-RAG ======================
# База знаний: статьи по базовым концепциям машинного обучения (ML glossary)

import warnings
warnings.filterwarnings('ignore')

import os
import random
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import faiss

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("=" * 80)
print("HW14 – Эмбеддинги, FAISS, оценка retrieval и mini-RAG")
print("=" * 80)

# ====================== 1. Seed и среда ======================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

os.makedirs("artifacts", exist_ok=True)
print(f"Seed: {SEED}")
print(f"faiss version: {faiss.__version__}\n")


# ====================== 2. База знаний и первичный анализ ======================

# База знаний: 20 коротких статей по концепциям машинного обучения.
# Каждый документ — это объяснение одного термина/метода ML.
# По ним имеет смысл строить RAG: пользователь задаёт вопрос про ML-концепцию,
# система ищет нужный документ и формирует ответ.

DOCUMENTS = [
    {
        "source": "doc_01_linear_regression",
        "text": "Linear regression is a supervised learning algorithm used to predict a continuous output variable based on one or more input features. It models the relationship between inputs and outputs as a linear function. The model is trained by minimizing the mean squared error (MSE) between predictions and actual values using gradient descent or the normal equation. Key assumptions include linearity, independence, homoscedasticity, and normally distributed errors."
    },
    {
        "source": "doc_02_logistic_regression",
        "text": "Logistic regression is a supervised classification algorithm that predicts the probability of a binary outcome. Despite its name, it is used for classification, not regression. It applies the sigmoid function to a linear combination of inputs to produce a probability between 0 and 1. The decision boundary is linear. Training uses binary cross-entropy loss and gradient descent. It can be extended to multiclass problems using softmax (multinomial logistic regression)."
    },
    {
        "source": "doc_03_decision_tree",
        "text": "A decision tree is a supervised learning algorithm that makes predictions by recursively splitting data based on feature values. Each internal node represents a feature test, each branch represents the outcome of the test, and each leaf node represents a class label or value. Trees are built by selecting splits that maximize information gain (for classification) or minimize variance (for regression). Decision trees are interpretable but prone to overfitting without pruning."
    },
    {
        "source": "doc_04_random_forest",
        "text": "Random forest is an ensemble learning method that builds multiple decision trees on bootstrapped subsets of training data and random subsets of features. Predictions are made by aggregating (majority vote for classification, averaging for regression) the outputs of all trees. This reduces overfitting compared to a single decision tree. Random forests are robust, handle high-dimensional data well, and provide feature importance scores."
    },
    {
        "source": "doc_05_svm",
        "text": "Support Vector Machine (SVM) is a supervised algorithm that finds the optimal hyperplane to separate classes by maximizing the margin between the nearest data points (support vectors) of each class. For non-linearly separable data, it uses the kernel trick to implicitly map data into a higher-dimensional space. Common kernels include linear, polynomial, and RBF (radial basis function). SVMs work well in high-dimensional spaces and are effective when the number of features exceeds the number of samples."
    },
    {
        "source": "doc_06_knn",
        "text": "K-Nearest Neighbors (KNN) is a non-parametric, lazy learning algorithm used for both classification and regression. It makes predictions based on the k closest training examples in the feature space. Distance metrics such as Euclidean, Manhattan, or cosine similarity are used to measure closeness. KNN is simple and interpretable but computationally expensive at prediction time and sensitive to irrelevant features and feature scale. Feature normalization is important."
    },
    {
        "source": "doc_07_neural_network",
        "text": "A neural network is a computational model inspired by the human brain, consisting of layers of interconnected nodes (neurons). Each connection has a weight that is learned during training. A typical feedforward neural network includes an input layer, one or more hidden layers, and an output layer. Activations functions like ReLU, sigmoid, or tanh introduce non-linearity. Training uses backpropagation and gradient descent to minimize a loss function."
    },
    {
        "source": "doc_08_cnn",
        "text": "Convolutional Neural Network (CNN) is a deep learning architecture designed for processing structured grid data like images. It uses convolutional layers to automatically learn spatial hierarchies of features through filters. Pooling layers reduce spatial dimensions and computation. CNNs have achieved state-of-the-art results in image classification, object detection, and segmentation tasks. Transfer learning with pretrained CNNs (e.g., ResNet, VGG) is widely used."
    },
    {
        "source": "doc_09_rnn",
        "text": "Recurrent Neural Network (RNN) is a neural network architecture designed for sequential data. Unlike feedforward networks, RNNs have connections that loop back, allowing them to maintain a hidden state that captures information from previous time steps. They are used in NLP, speech recognition, and time series tasks. Vanilla RNNs suffer from vanishing gradients; LSTM (Long Short-Term Memory) and GRU (Gated Recurrent Unit) were developed to address this problem."
    },
    {
        "source": "doc_10_transformer",
        "text": "The Transformer is a deep learning architecture introduced in the paper 'Attention is All You Need' (Vaswani et al., 2017). It relies entirely on self-attention mechanisms to model relationships between all positions in a sequence simultaneously, without recurrence. The architecture consists of an encoder and a decoder, each made up of multi-head attention layers and feed-forward layers. Transformers are the foundation of modern NLP models like BERT, GPT, and T5."
    },
    {
        "source": "doc_11_overfitting",
        "text": "Overfitting occurs when a machine learning model learns the training data too well, including its noise and random fluctuations, leading to poor generalization on unseen data. Signs of overfitting include low training loss but high validation loss. Common remedies include regularization (L1/L2), dropout, data augmentation, early stopping, reducing model complexity, and collecting more training data. Cross-validation helps detect overfitting."
    },
    {
        "source": "doc_12_regularization",
        "text": "Regularization is a technique used to prevent overfitting by adding a penalty term to the loss function. L1 regularization (Lasso) adds the sum of absolute values of weights, encouraging sparsity. L2 regularization (Ridge) adds the sum of squared weights, encouraging small weights. Elastic Net combines both L1 and L2 penalties. Dropout is a regularization technique for neural networks that randomly deactivates neurons during training."
    },
    {
        "source": "doc_13_cross_validation",
        "text": "Cross-validation is a model evaluation technique that partitions data into subsets (folds) to assess how a model generalizes to an independent dataset. In k-fold cross-validation, the data is split into k equal parts; the model is trained on k-1 folds and tested on the remaining fold, repeating k times. This provides a more reliable estimate of model performance than a single train-test split. Stratified k-fold preserves class distribution across folds."
    },
    {
        "source": "doc_14_gradient_descent",
        "text": "Gradient descent is an iterative optimization algorithm used to minimize a loss function by updating model parameters in the direction of the negative gradient. Batch gradient descent computes the gradient over the entire dataset. Stochastic gradient descent (SGD) uses a single sample per update. Mini-batch gradient descent uses a small subset. Adaptive optimizers like Adam, RMSprop, and Adagrad adjust the learning rate for each parameter automatically."
    },
    {
        "source": "doc_15_clustering",
        "text": "Clustering is an unsupervised learning task that groups data points into clusters based on similarity, without using labeled examples. K-Means assigns each point to the nearest cluster centroid and iteratively updates centroids. DBSCAN groups points based on density and can find clusters of arbitrary shape. Hierarchical clustering builds a tree of clusters (dendrogram). Evaluation metrics for clustering include silhouette score and Davies-Bouldin index."
    },
    {
        "source": "doc_16_pca",
        "text": "Principal Component Analysis (PCA) is an unsupervised dimensionality reduction technique that projects data onto a lower-dimensional space by finding the directions (principal components) of maximum variance. It uses eigendecomposition of the covariance matrix. PCA is used for visualization, noise reduction, and as a preprocessing step. The number of components to retain is often chosen based on the explained variance ratio (e.g., 95% of total variance)."
    },
    {
        "source": "doc_17_attention",
        "text": "Attention mechanism in neural networks allows the model to focus on different parts of the input when producing each part of the output. Scaled dot-product attention computes a weighted sum of values, where weights are determined by the compatibility of queries and keys. Multi-head attention runs multiple attention functions in parallel and concatenates the results. Self-attention relates different positions of a single sequence to compute its representation."
    },
    {
        "source": "doc_18_embeddings",
        "text": "Embeddings are dense vector representations of discrete objects (words, sentences, users, items) in a continuous vector space. Word embeddings like Word2Vec and GloVe capture semantic relationships so that similar words have similar vectors. Sentence embeddings represent entire sentences as single vectors. Embeddings are the foundation of modern NLP: they allow neural networks to work with text, and enable semantic search, recommendation, and clustering tasks."
    },
    {
        "source": "doc_19_transfer_learning",
        "text": "Transfer learning is a technique where a model trained on one task is reused as the starting point for a model on a different but related task. In NLP, pretrained language models (BERT, GPT) are fine-tuned on downstream tasks. In computer vision, CNNs pretrained on ImageNet are fine-tuned for specific datasets. Transfer learning significantly reduces the amount of labeled data and training time required for the target task."
    },
    {
        "source": "doc_20_evaluation_metrics",
        "text": "Evaluation metrics measure the performance of machine learning models. For classification: accuracy (fraction of correct predictions), precision (fraction of positive predictions that are correct), recall (fraction of actual positives correctly identified), F1-score (harmonic mean of precision and recall), and AUC-ROC (area under the ROC curve). For regression: MSE (mean squared error), RMSE (root MSE), MAE (mean absolute error), and R-squared. Choosing the right metric depends on the problem and class imbalance."
    }
]

print(f"Число документов в базе знаний: {len(DOCUMENTS)}")
print("\nПредметная область: концепции и алгоритмы машинного обучения.")
print("Это справочная база знаний по ML: каждый документ объясняет один термин/метод.")
print("По ней разумно строить RAG: пользователь задаёт вопрос про ML-концепцию,")
print("система ищет нужный документ и формирует ответ на основе найденного контекста.\n")

print("=== Примеры документов ===")
for doc in DOCUMENTS[:4]:
    print(f"\n[{doc['source']}]")
    print(doc['text'][:200] + "...")


# ====================== 3. Чанкинг документов ======================

CHUNK_SIZE = 200   # символов на чанк
CHUNK_OVERLAP = 40  # символов перекрытия

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Разбиение текста на перекрывающиеся фрагменты по символам."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(text):
            break
        start += chunk_size - overlap
    return chunks

# Создаём чанки для всех документов
all_chunks = []
for doc in DOCUMENTS:
    chunks = chunk_text(doc["text"])
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "source": doc["source"],
            "chunk_id": f"{doc['source']}_chunk{i}",
            "text": chunk
        })

print(f"Число чанков после разбиения: {len(all_chunks)}")
print(f"Параметры чанкинга: chunk_size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}")
print("Выбор: символьный чанкинг с перекрытием — простой и воспроизводимый.")
print("Перекрытие уменьшает вероятность потери контекста на границах чанков.\n")

# Пример чанкинга одного документа
example_doc = DOCUMENTS[0]
example_chunks = chunk_text(example_doc["text"])
print(f"=== Пример чанкинга документа [{example_doc['source']}] ===")
for i, ch in enumerate(example_chunks):
    print(f"\nЧанк {i}: '{ch[:100]}...'")


# ====================== 4. Эмбеддинги и индекс FAISS ======================

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

print(f"Загрузка embedding-модели: {EMBEDDING_MODEL_NAME}...")
embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print("Модель загружена.\n")

def get_embeddings(texts, model):
    return model.encode(texts, normalize_embeddings=True, show_progress_bar=False)

# Векторизация чанков
chunk_texts = [c["text"] for c in all_chunks]
print("Вычисление эмбеддингов для чанков...")
chunk_embeddings = get_embeddings(chunk_texts, embed_model)
print(f"Форма матрицы эмбеддингов: {chunk_embeddings.shape}")

# Построение FAISS-индекса (косинусное сходство через нормализованные векторы + IndexFlatIP)
DIM = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(DIM)  # Inner Product = косинусное сходство для нормализованных векторов
index.add(chunk_embeddings.astype(np.float32))
print(f"FAISS-индекс построен. Тип: IndexFlatIP, размерность: {DIM}, векторов: {index.ntotal}\n")

TOP_K = 3

def retrieve(query, index, chunks, model, top_k=TOP_K):
    q_emb = get_embeddings([query], model).astype(np.float32)
    scores, indices = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "chunk_id": chunks[idx]["chunk_id"],
            "source": chunks[idx]["source"],
            "text": chunks[idx]["text"],
            "score": float(score)
        })
    return results

# Демонстрация поиска
demo_queries = [
    "What is gradient descent and how does it work?",
    "How does attention mechanism work in transformers?",
    "What is overfitting and how to prevent it?",
    "How does random forest differ from decision tree?",
    "What are word embeddings?"
]

print("=== Демонстрация retrieval ===")
for q in demo_queries:
    results = retrieve(q, index, all_chunks, embed_model)
    print(f"\nЗапрос: {q}")
    for r in results:
        print(f"  [{r['source']}] score={r['score']:.3f} | {r['text'][:80]}...")


# ====================== 5. Контрольные запросы и оценка retrieval ======================

CONTROL_QUERIES = [
    {"query": "What is linear regression used for?",         "expected_source": "doc_01_linear_regression"},
    {"query": "How does logistic regression classify data?",   "expected_source": "doc_02_logistic_regression"},
    {"query": "How is a decision tree built?",                 "expected_source": "doc_03_decision_tree"},
    {"query": "What makes random forest better than a single tree?", "expected_source": "doc_04_random_forest"},
    {"query": "What is the kernel trick in SVM?",             "expected_source": "doc_05_svm"},
    {"query": "How does gradient descent optimize models?",   "expected_source": "doc_14_gradient_descent"},
    {"query": "What is overfitting and how to prevent it?",   "expected_source": "doc_11_overfitting"},
    {"query": "What is regularization in machine learning?",  "expected_source": "doc_12_regularization"},
    {"query": "How does PCA reduce dimensionality?",          "expected_source": "doc_16_pca"},
    {"query": "What is transfer learning?",                   "expected_source": "doc_19_transfer_learning"},
    {"query": "How do sentence embeddings work?",             "expected_source": "doc_18_embeddings"},
    {"query": "What metrics are used to evaluate classifiers?", "expected_source": "doc_20_evaluation_metrics"},
]

eval_rows = []
for cq in CONTROL_QUERIES:
    results = retrieve(cq["query"], index, all_chunks, embed_model, top_k=TOP_K)
    retrieved_sources = [r["source"] for r in results]
    hit = 1 if cq["expected_source"] in retrieved_sources else 0
    # rank of first relevant
    rank = None
    for i, src in enumerate(retrieved_sources, 1):
        if src == cq["expected_source"]:
            rank = i
            break
    eval_rows.append({
        "query": cq["query"],
        "expected_source": cq["expected_source"],
        "retrieved_sources": " | ".join(retrieved_sources),
        "hit_at_k": hit,
        "rank_of_first_relevant": rank if rank is not None else -1
    })

eval_df = pd.DataFrame(eval_rows)

hit_at_k = eval_df["hit_at_k"].mean()
# recall@k: для каждого запроса — доля релевантных в top-k (у нас 1 релевантный → = hit@k)
recall_at_k = hit_at_k  # 1 expected per query
# MRR@k
mrr = np.mean([1.0 / r if r > 0 else 0.0 for r in eval_df["rank_of_first_relevant"]])

print(f"=== Оценка retrieval (top_k={TOP_K}) ===")
print(f"Число контрольных запросов: {len(CONTROL_QUERIES)}")
print(f"hit@{TOP_K}    = {hit_at_k:.3f}")
print(f"recall@{TOP_K} = {recall_at_k:.3f}")
print(f"MRR@{TOP_K}    = {mrr:.3f}\n")

display(eval_df)

# Сохранение артефакта
eval_df.to_csv("artifacts/retrieval_eval.csv", index=False)
print("\nСохранено: artifacts/retrieval_eval.csv")

# Анализ
easy = eval_df[eval_df["hit_at_k"] == 1]["query"].tolist()
hard = eval_df[eval_df["hit_at_k"] == 0]["query"].tolist()
print(f"\nЛёгкие запросы (hit=1): {easy}")
print(f"Проблемные запросы (hit=0): {hard if hard else 'нет'}")


# ====================== 6. Эксперимент: сравнение top_k ======================

print("=== Эксперимент: сравнение top_k=3 vs top_k=5 ===")

results_by_k = {}
for k in [3, 5]:
    hits = []
    recalls = []
    for cq in CONTROL_QUERIES:
        results = retrieve(cq["query"], index, all_chunks, embed_model, top_k=k)
        retrieved_sources = [r["source"] for r in results]
        hit = 1 if cq["expected_source"] in retrieved_sources else 0
        hits.append(hit)
        recalls.append(hit)  # 1 relevant per query
    results_by_k[k] = {
        "hit@k": np.mean(hits),
        "recall@k": np.mean(recalls)
    }

comparison_df = pd.DataFrame([
    {"top_k": k, "hit@k": v["hit@k"], "recall@k": v["recall@k"]}
    for k, v in results_by_k.items()
])
print(comparison_df.to_string(index=False))

print(f"""
Вывод: top_k=5 даёт hit@k={results_by_k[5]['hit@k']:.3f} vs top_k=3 → {results_by_k[3]['hit@k']:.3f}.
При top_k=5 шанс найти релевантный документ выше, но контекст для RAG становится шире.
Для mini-RAG оставляем top_k=3 как компромисс между качеством и краткостью контекста.
""")


# ====================== 7. Обновление базы знаний и переиндексация ======================

NEW_DOCUMENTS = [
    {
        "source": "doc_21_bagging",
        "text": "Bagging (Bootstrap Aggregating) is an ensemble meta-algorithm that trains multiple models on different random subsets of the training data (with replacement) and aggregates their predictions. It reduces variance without significantly increasing bias. Random forest is the most well-known bagging method. Bagging works particularly well with high-variance, low-bias models like decision trees."
    },
    {
        "source": "doc_22_boosting",
        "text": "Boosting is an ensemble method that combines weak learners sequentially, where each model focuses on the mistakes of the previous one. Popular boosting algorithms include AdaBoost, Gradient Boosting, XGBoost, LightGBM, and CatBoost. Boosting reduces both bias and variance. It can overfit on noisy data. Gradient boosting minimizes a loss function by adding trees that approximate the negative gradient of the loss."
    },
    {
        "source": "doc_23_batch_norm",
        "text": "Batch normalization is a technique used in neural networks to normalize the inputs of each layer to have zero mean and unit variance across the mini-batch. It accelerates training, allows higher learning rates, reduces the dependence on careful initialization, and has a slight regularization effect. Batch norm is applied before or after the activation function. Layer normalization is an alternative used in Transformers and recurrent networks."
    },
    {
        "source": "doc_24_dropout",
        "text": "Dropout is a regularization technique for neural networks where randomly selected neurons are ignored (dropped out) during training with a given probability p. This prevents neurons from co-adapting and forces the network to learn robust features. At inference time, all neurons are used and their outputs are scaled by (1-p). Dropout is commonly applied to fully connected layers and is less common in convolutional layers."
    },
    {
        "source": "doc_25_rag",
        "text": "Retrieval-Augmented Generation (RAG) is a framework that combines information retrieval with text generation. Given a user query, RAG first retrieves relevant documents or passages from a knowledge base using a retrieval system (e.g., FAISS with dense embeddings), then passes both the query and retrieved context to a generative model to produce an answer. RAG allows models to access up-to-date information without retraining and provides verifiable sources."
    }
]

# Запросы для сравнения до/после
update_test_queries = [
    {"query": "What is bagging in ensemble learning?",          "expected_source": "doc_21_bagging"},
    {"query": "How does boosting differ from bagging?",         "expected_source": "doc_22_boosting"},
    {"query": "What is batch normalization?",                    "expected_source": "doc_23_batch_norm"},
    {"query": "How does dropout prevent overfitting?",           "expected_source": "doc_24_dropout"},
    {"query": "What is Retrieval-Augmented Generation (RAG)?",  "expected_source": "doc_25_rag"},
]

# Retrieval ДО обновления
before_after_rows = []
for q in update_test_queries:
    results_before = retrieve(q["query"], index, all_chunks, embed_model, top_k=TOP_K)
    before_sources = " | ".join([r["source"] for r in results_before])
    before_after_rows.append({"query": q["query"], "before": before_sources, "expected": q["expected_source"]})

# Обновляем базу знаний
all_chunks_updated = list(all_chunks)
for doc in NEW_DOCUMENTS:
    chunks = chunk_text(doc["text"])
    for i, chunk in enumerate(chunks):
        all_chunks_updated.append({
            "source": doc["source"],
            "chunk_id": f"{doc['source']}_chunk{i}",
            "text": chunk
        })

print(f"Чанков до обновления: {len(all_chunks)}")
print(f"Новых документов добавлено: {len(NEW_DOCUMENTS)}")
print(f"Чанков после обновления: {len(all_chunks_updated)}")

# Переиндексация
chunk_texts_updated = [c["text"] for c in all_chunks_updated]
print("\nПереиндексация (вычисление эмбеддингов для всех чанков)...")
chunk_embeddings_updated = get_embeddings(chunk_texts_updated, embed_model)
index_updated = faiss.IndexFlatIP(DIM)
index_updated.add(chunk_embeddings_updated.astype(np.float32))
print(f"Новый индекс: {index_updated.ntotal} векторов\n")

# Retrieval ПОСЛЕ обновления
final_rows = []
for row, q in zip(before_after_rows, update_test_queries):
    results_after = retrieve(q["query"], index_updated, all_chunks_updated, embed_model, top_k=TOP_K)
    after_sources = " | ".join([r["source"] for r in results_after])
    changed = int(row["before"] != after_sources)
    final_rows.append({
        "query": row["query"],
        "before_retrieved_sources": row["before"],
        "after_retrieved_sources": after_sources,
        "changed": changed
    })

before_after_df = pd.DataFrame(final_rows)
display(before_after_df)

before_after_df.to_csv("artifacts/retrieval_before_after_update.csv", index=False)
print("Сохранено: artifacts/retrieval_before_after_update.csv")

print(f"\nЗапросов, где retrieval изменился после обновления: {before_after_df['changed'].sum()} из {len(final_rows)}")
print("Вывод: после добавления новых документов модель стала находить релевантные источники")
print("по запросам про bagging, boosting, batch norm, dropout и RAG — которых раньше не было.")


# ====================== 8. Mini-RAG ======================

def build_context(results):
    """Собирает контекст из найденных чанков."""
    parts = []
    for i, r in enumerate(results, 1):
        parts.append(f"[{i}] (source: {r['source']})\n{r['text']}")
    return "\n\n".join(parts)

def simple_answer(query, context, sources):
    """
    Шаблонный генератор ответа: извлекает наиболее релевантный чанк из контекста.
    В учебном варианте — выбираем лучший чанк по score и формируем ответ-цитату с пояснением.
    """
    best_chunk = sources[0]
    answer = (
        f"На основе найденных документов:\n"
        f"{best_chunk['text']}\n\n"
        f"(Если нужно больше деталей, см. также: {', '.join([s['source'] for s in sources[1:]])})"
    )
    return answer

def mini_rag(query, index, chunks, model, top_k=TOP_K):
    results = retrieve(query, index, chunks, model, top_k=top_k)
    context = build_context(results)
    answer = simple_answer(query, context, results)
    retrieved_sources = [r["source"] for r in results]
    return {
        "question": query,
        "answer": answer,
        "retrieved_sources": " | ".join(retrieved_sources),
        "context": context
    }

RAG_QUESTIONS = [
    "What is gradient descent and how does it work?",
    "How does the attention mechanism work in transformers?",
    "What is cross-validation and why is it useful?",
    "How does dropout prevent overfitting in neural networks?",
    "What is Retrieval-Augmented Generation?",
    "How does K-Means clustering work?",
    "What is the difference between precision and recall?",
    "How does PCA reduce the number of dimensions?",
]

rag_results = []
print("=== Mini-RAG — ответы на вопросы ===\n")
for question in RAG_QUESTIONS:
    result = mini_rag(question, index_updated, all_chunks_updated, embed_model)
    rag_results.append({
        "question": result["question"],
        "answer": result["answer"][:300] + "...",
        "retrieved_sources": result["retrieved_sources"]
    })
    print(f"Q: {question}")
    print(f"Sources: {result['retrieved_sources']}")
    print(f"A: {result['answer'][:200]}...")
    print()

rag_df = pd.DataFrame(rag_results)
rag_df.to_csv("artifacts/rag_examples.csv", index=False)
print("Сохранено: artifacts/rag_examples.csv")


# ====================== 9. Краткий анализ ошибок ======================

print("=== Краткий анализ ошибок и пограничных случаев mini-RAG ===\n")

error_cases = [
    "What is the difference between L1 and L2 regularization?",
    "How does KNN handle high-dimensional data?",
    "What is the vanishing gradient problem?",
    "Why does bagging reduce variance?"
]

for q in error_cases:
    result = mini_rag(q, index_updated, all_chunks_updated, embed_model)
    print(f"Q: {q}")
    print(f"Sources: {result['retrieved_sources']}")
    print(f"A: {result['answer'][:200]}...")
    print()

print("Анализ:")
print("1. 'L1 vs L2 regularization' — оба понятия описаны в одном документе (doc_12_regularization).")
print("   Retrieval находит нужный документ, но ответ содержит сразу обе концепции без разграничения.")
print("   Ошибка: контекст не структурирован — шаблонный генератор не разделяет ответ на части.")
print()
print("2. 'KNN and high-dimensional data' — doc_06_knn упоминает проблему масштаба признаков,")
print("   но не объясняет 'curse of dimensionality' детально. Информация в базе неполная.")
print()
print("3. 'Vanishing gradient problem' — это понятие упоминается вскользь в doc_09_rnn,")
print("   но нет отдельного документа. Retrieval может вернуть doc_09_rnn или doc_07_neural_network,")
print("   ответ будет частичным. Проблема: неполнота базы знаний.")
print()
print("4. 'Bagging reduces variance' — после обновления базы doc_21_bagging даёт хороший ответ.")
print("   ДО обновления базы этот запрос не находил релевантного документа — наглядный эффект переиндексации.")




HW14 – Эмбеддинги, FAISS, оценка retrieval и mini-RAG
Seed: 42
faiss version: 1.13.2

Число документов в базе знаний: 20

Предметная область: концепции и алгоритмы машинного обучения.
Это справочная база знаний по ML: каждый документ объясняет один термин/метод.
По ней разумно строить RAG: пользователь задаёт вопрос про ML-концепцию,
система ищет нужный документ и формирует ответ на основе найденного контекста.

=== Примеры документов ===

[doc_01_linear_regression]
Linear regression is a supervised learning algorithm used to predict a continuous output variable based on one or more input features. It models the relationship between inputs and outputs as a linear...

[doc_02_logistic_regression]
Logistic regression is a supervised classification algorithm that predicts the probability of a binary outcome. Despite its name, it is used for classification, not regression. It applies the sigmoid ...

[doc_03_decision_tree]
A decision tree is a supervised learning algorithm that makes predi

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Модель загружена.

Вычисление эмбеддингов для чанков...
Форма матрицы эмбеддингов: (60, 384)
FAISS-индекс построен. Тип: IndexFlatIP, размерность: 384, векторов: 60

=== Демонстрация retrieval ===

Запрос: What is gradient descent and how does it work?
  [doc_14_gradient_descent] score=0.779 | Gradient descent is an iterative optimization algorithm used to minimize a loss ...
  [doc_14_gradient_descent] score=0.643 | Batch gradient descent computes the gradient over the entire dataset. Stochastic...
  [doc_12_regularization] score=0.477 | Regularization is a technique used to prevent overfitting by adding a penalty te...

Запрос: How does attention mechanism work in transformers?
  [doc_10_transformer] score=0.662 | The Transformer is a deep learning architecture introduced in the paper 'Attenti...
  [doc_10_transformer] score=0.603 | , each made up of multi-head attention layers and feed-forward layers. Transform...
  [doc_17_attention] score=0.512 | Attention mechanism in neural netw

,query,expected_source,retrieved_sources,hit_at_k,rank_of_first_relevant
0,What is linear regression used for?,doc_01_linear_regression,doc_01_linear_regression | doc_02_logistic_reg...,1,1
1,How does logistic regression classify data?,doc_02_logistic_regression,doc_02_logistic_regression | doc_02_logistic_r...,1,1
2,How is a decision tree built?,doc_03_decision_tree,doc_03_decision_tree | doc_03_decision_tree | ...,1,1
3,What makes random forest better than a single ...,doc_04_random_forest,doc_04_random_forest | doc_04_random_forest | ...,1,1
4,What is the kernel trick in SVM?,doc_05_svm,doc_05_svm | doc_05_svm | doc_05_svm,1,1
5,How does gradient descent optimize models?,doc_14_gradient_descent,doc_14_gradient_descent | doc_14_gradient_desc...,1,1
6,What is overfitting and how to prevent it?,doc_11_overfitting,doc_11_overfitting | doc_11_overfitting | doc_...,1,1
7,What is regularization in machine learning?,doc_12_regularization,doc_12_regularization | doc_12_regularization ...,1,1
8,How does PCA reduce dimensionality?,doc_16_pca,doc_16_pca | doc_16_pca | doc_06_knn,1,1
9,What is transfer learning?,doc_19_transfer_learning,doc_19_transfer_learning | doc_08_cnn | doc_19...,1,1



Сохранено: artifacts/retrieval_eval.csv

Лёгкие запросы (hit=1): ['What is linear regression used for?', 'How does logistic regression classify data?', 'How is a decision tree built?', 'What makes random forest better than a single tree?', 'What is the kernel trick in SVM?', 'How does gradient descent optimize models?', 'What is overfitting and how to prevent it?', 'What is regularization in machine learning?', 'How does PCA reduce dimensionality?', 'What is transfer learning?', 'How do sentence embeddings work?', 'What metrics are used to evaluate classifiers?']
Проблемные запросы (hit=0): нет
=== Эксперимент: сравнение top_k=3 vs top_k=5 ===
 top_k  hit@k  recall@k
     3    1.0       1.0
     5    1.0       1.0

Вывод: top_k=5 даёт hit@k=1.000 vs top_k=3 → 1.000.
При top_k=5 шанс найти релевантный документ выше, но контекст для RAG становится шире.
Для mini-RAG оставляем top_k=3 как компромисс между качеством и краткостью контекста.

Чанков до обновления: 60
Новых документов добавл

,query,before_retrieved_sources,after_retrieved_sources,changed
0,What is bagging in ensemble learning?,doc_04_random_forest | doc_02_logistic_regress...,doc_21_bagging | doc_21_bagging | doc_04_rando...,1
1,How does boosting differ from bagging?,doc_04_random_forest | doc_03_decision_tree | ...,doc_21_bagging | doc_22_boosting | doc_21_bagging,1
2,What is batch normalization?,doc_14_gradient_descent | doc_14_gradient_desc...,doc_23_batch_norm | doc_23_batch_norm | doc_23...,1
3,How does dropout prevent overfitting?,doc_12_regularization | doc_11_overfitting | d...,doc_12_regularization | doc_24_dropout | doc_2...,1
4,What is Retrieval-Augmented Generation (RAG)?,doc_19_transfer_learning | doc_09_rnn | doc_17...,doc_25_rag | doc_25_rag | doc_19_transfer_lear...,1


Сохранено: artifacts/retrieval_before_after_update.csv

Запросов, где retrieval изменился после обновления: 5 из 5
Вывод: после добавления новых документов модель стала находить релевантные источники
по запросам про bagging, boosting, batch norm, dropout и RAG — которых раньше не было.
=== Mini-RAG — ответы на вопросы ===

Q: What is gradient descent and how does it work?
Sources: doc_14_gradient_descent | doc_14_gradient_descent | doc_12_regularization
A: На основе найденных документов:
Gradient descent is an iterative optimization algorithm used to minimize a loss function by updating model parameters in the direction of the negative gradient. Batch g...

Q: How does the attention mechanism work in transformers?
Sources: doc_10_transformer | doc_10_transformer | doc_17_attention
A: На основе найденных документов:
The Transformer is a deep learning architecture introduced in the paper 'Attention is All You Need' (Vaswani et al., 2017). It relies entirely on self-attention mechani...

